# Using the DynamicFunction Module in baseobjects

## Introduction

DynamicFunction is an abstract function class with multiplexed binding and callback. It inherits DynamicCallable and BaseFunction and uses DynamicMethod as its method_type, enabling method-like behavior when used as a descriptor while retaining function semantics.

This tutorial demonstrates:
- Creating and using DynamicFunction
- Switching call strategies via MethodMultiplexer
- Descriptor binding and interaction with DynamicMethod
- Converting between function and method semantics in dynamic contexts

**Prerequisites:**
- Familiarity with BaseCallable and DynamicCallable
- Understanding of the descriptor protocol and class descriptors

### Table of Contents
- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting--FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [8]:
from baseobjects.functions import DynamicFunction

## Core Functionality

We'll build a small DynamicFunction subclass that supports multiple call strategies via the call_multiplexer, and show binding behavior when placed on a class.

In [9]:
class MathFunc(DynamicFunction):
    """A small DynamicFunction with multiple call strategies."""

    def call_wrapped(self, a, b):
        return a + b

    def call_multiply(self, a, b):
        return a * b

# Create the function-like object
mf = MathFunc()
print("Default call (wrapped):", mf(2, 3))

# Switch call strategy
mf.call_method = "call_multiply"
print("Multiply strategy:", mf(2, 3))

Default call (wrapped): 5
Multiply strategy: 6


### Method Multiplexer Highlight: Binding and Callback

- call_multiplexer determines which method is invoked when the object is called.
- bind_multiplexer determines how the object binds when accessed as a descriptor (via __get__).

Let's place create MathFunc as a method in a class to show binding behavior.

In [10]:
class MathMethod(DynamicFunction):
    """A small DynamicFunction with multiple bind strategies."""

    def call_wrapped(self, instance, a, b):
        return a + b

    # Binding strategies
    def bind_builtin(self, instance=None, owner=None):
        # default binding behavior mimicking BaseCallable.bind_builtin
        print("bind builtin with:", instance, owner)
        return super().bind_builtin(instance=instance, owner=owner)

    def bind(self, instance=None, owner=None):
        # provide an alternate binding path returning a bound method to instance
        print("bind (normal) with:", instance, owner)
        return super().bind(instance=instance, owner=owner)


class Container:
    op = MathMethod()


c = Container()
print("Bound call (default):", c.op(4, 5))

# Change the strategy globally for the descriptor
Container.op.bind_method = "bind"
print("Bound call (bind):", c.op(4, 5))

bind builtin with: <__main__.Container object at 0x0000029E99430260> <class '__main__.Container'>
Bound call (default): 9
bind builtin with: None <class '__main__.Container'>
bind (normal) with: <__main__.Container object at 0x0000029E99430260> <class '__main__.Container'>
Bound call (bind): 9


## Module Interaction

DynamicFunction declares method_type = DynamicMethod, which means when used as a descriptor in classes, it binds to instances using method semantics shaped by DynamicMethod. It inherits DynamicCallable's pickling of multiplexer state.

In [11]:
import pickle
p = pickle.dumps(Container.op)
r = pickle.loads(p)
print("Restored call method:", r.call_method)

bind (normal) with: None <class '__main__.Container'>
Restored call method: call_wrapped


## Advanced Features

You can create variants that don't rely on a wrapped function at all; register and select custom strategies.

In [12]:
class NonWrappingDynamicFunction(DynamicFunction):
    def __init__(self, *args, **kwargs):
        super().__init__(None, *args, **kwargs)
        self.call_method = "add"
    def add(self, a, b):
        return a + b
    def subtract(self, a, b):
        return a - b
    def multiply(self, a, b):
        return a * b

nf = NonWrappingDynamicFunction()
print("Add:", nf(7, 2))
nf.call_method = "multiply"
print("Multiply:", nf(7, 2))

Add: 9
Multiply: 14


## Examples

- Toggle call strategies for validation/logging without changing call sites.
- Use in classes to provide adaptable behavior per environment.

In [13]:
class LoggerFunction(DynamicFunction):
    def call_wrapped(self, *args, **kwargs):
        return sum(args)
    def call_with_logging(self, *args, **kwargs):
        print("[log]", args, kwargs)
        return sum(args)
    def call_other_logging(self, *args, **kwargs):
        print("[loging 2]", args, kwargs)
        return sum(args)

lf = LoggerFunction()
print(lf(1,2,3))
lf.call_method = "call_other_logging"
print(lf(1,2,3))
lf.call_method = "call_with_logging"
print(lf(1,2,3))

6
[loging 2] (1, 2, 3) {}
6
[log] (1, 2, 3) {}
6


## API Highlights

- Inherits: DynamicCallable, BaseFunction
- method_type = DynamicMethod
- Properties: bind_method, call_method
- Multiplexer attributes: bind_multiplexer, call_multiplexer
- __get__ delegates binding; __call__ delegates invocation

## Troubleshooting / FAQs

### Q: Why does using on a class behave like a method?

A: DynamicFunction uses DynamicMethod as its method_type, so descriptor binding follows method semantics while retaining function-like usage when standalone.

### Q: How do I define new strategies?

A: Implement methods on your subclass and select them by name via call_method or bind_method.

## Conclusion and Next Steps

DynamicFunction brings MethodMultiplexer-driven binding and callback selection to function-like objects. See also DynamicMethod and MethodMultiplexer tutorials for deeper control.
